# TDA replication on Google Colab

Reproduction of **Karmanov et al., *Efficient Test-Time Adaptation of Vision-Language Models*, CVPR 2024**
([paper](https://openaccess.thecvf.com/content/CVPR2024/html/Karmanov_Efficient_Test-Time_Adaptation_of_Vision-Language_Models_CVPR_2024_paper.html),
[official code](https://github.com/kdiAAA/TDA)).

This notebook is a thin driver: it clones this repository and runs **the same files as the cluster**
(`prepare_data.py`, `experiments.py`, `tda_replica.py`, `experiments.yaml`). Nothing is duplicated here,
so a change you push to the repository is used the next time you run the notebook.

**How to use it**
1. *Runtime → Change runtime type → GPU* (the free T4 is enough).
2. In **Settings** below, set `REPO_URL` to your GitHub repository and choose datasets and experiments.
3. *Runtime → Run all*. Results, logs and small dataset archives are saved to Google Drive
   (`MyDrive/TDA_replication/`), so a new session continues where the previous one stopped.
4. Plots and stream analysis: open `notebooks/analysis.ipynb` (same Drive folder).

**Colab vs cluster.** Colab uses its own PyTorch version (not the authors' 1.12.1) and sessions last a few
hours: fine for the cross-domain datasets, too short for the OOD benchmark (64 augmented views per image).

## Background: the TDA method

### 1.1 Zero-shot CLIP
For an $N$-class problem, every class name is inserted into hand-crafted prompts (e.g. *"a photo of a {class}."*) and encoded by the text encoder $E_t$. The normalised embeddings, averaged over the prompts, form the classifier $W_c \in \mathbb{R}^{N \times C}$. A test image $x$ is encoded by the image encoder $E_v$ into a normalised feature $f = E_v(x)$, and the zero-shot logits are

$$P_{\text{CLIP}}(f) = 100 \cdot f\,W_c^{\top}.$$

### 1.2 Test-time adaptation setting
Test images arrive **one at a time** (batch size 1) and without labels. The model can use only the samples it has already seen. TDA does this **without any backpropagation**: it keeps two small key–value caches and reads them out with a Tip-Adapter style affinity function

$$\mathcal{A}(z) = \alpha \exp\big(-\beta (1 - z)\big),$$

where $\alpha$ is a residual weight and $\beta$ a sharpness ratio.

### 1.3 Positive cache
- **Keys** $Q_p$: features of test samples. **Values** $\hat{L}_p$: their one-hot pseudo-labels (the CLIP prediction).
- Each class has its own queue with a maximum *shot capacity* $k$ (default 3). A new sample predicted as class $c$ is added if the queue of $c$ is not full. Otherwise it replaces the stored sample with the **highest entropy**, but only if its own entropy is lower. The cache thus keeps the most confident examples of each class.
- Read-out: $P_{\text{pos}}(f) = \mathcal{A}\big(f\,Q_p^{\top}\big)\,\hat{L}_p.$

### 1.4 Negative cache (negative pseudo-labelling)
- A sample enters the negative cache only if its prediction is **moderately uncertain**: $\tau_l < H(f W_c^{\top}) < \tau_h$, with normalised entropy thresholds $[\tau_l, \tau_h] = [0.2, 0.5]$. Shot capacity $\tilde{k} = 2$.
- Its value is a **negative pseudo-label**: the set of classes with probability above $p_l = 0.03$, i.e. the classes the image is still confused with.
- The read-out lowers the logits of those classes: $P_{\text{neg}}(f) = -\,\mathcal{A}\big(f\,Q_n^{\top}\big)\,\mathbb{1}\big[p_l < P(Q_n) < 1\big].$

### 1.5 Final prediction
$$P_{\text{TDA}}(f) = P_{\text{CLIP}}(f) + P_{\text{pos}}(f) + P_{\text{neg}}(f).$$

### 1.6 Augmented views for the OOD datasets
For ImageNet-A/V/R/S each test image is expanded into 1 original + 63 AugMix views. Only the 10% of views with the lowest entropy (6 views) are kept, as in TPT, and their features and logits are averaged. ImageNet and the 10 cross-domain datasets use a single view.

### 1.7 Paper vs code
The paper is the description; the official code is the reference implementation. Where they differ, this repository follows the code:
- **Sign of the negative cache**: Eq. 4 defines $\hat L_n$ with a $-1$ and Eq. 6 adds another minus sign, so the two cancel. The code *subtracts* $\mathcal{A}(f Q_n^\top)\,\mathbb{1}[p_l < p < 1]$, which is the form written in 1.4.
- **Entropy normalisation**: entropies are computed in nats and divided by $\log_2 N$ before being compared with $[\tau_l, \tau_h]$.
- **Averaged view feature**: the mean of the selected views is not re-normalised.
- **Hyper-parameters**: $\alpha$, $\beta$, shot capacities and thresholds are read per dataset from `configs/*.yaml`.

### 1.8 Algorithm (pseudo-code)
```
for each test image x (stream, batch size 1):
    f, logits, H, p, ŷ = CLIP(x)                       # optional multi-view selection
    positive_cache.update(ŷ, key=f, priority=H, value=onehot(ŷ))
    if τ_l < H / log2(N) < τ_h:
        negative_cache.update(ŷ, key=f, priority=H, value=p)
    logits += A_pos(f · Q_pᵀ) · L_p
    logits -= A_neg(f · Q_nᵀ) · 1[p_l < P_n < 1]
    prediction = argmax(logits)
```

## 1. GPU
The official code calls `.cuda()` everywhere: this must show a GPU.

In [ ]:
!nvidia-smi

## 2. Settings
- `REPO_URL`: your copy of this repository. If it is private, use
  `https://<token>@github.com/<user>/TDA_replication.git` with a GitHub token (do not share the notebook then).
- `DATASETS`: what to download. Small ones first; names as in `experiments.yaml`
  (`prepare_data.py` uses `imagenet-a`, `imagenet-r`, `imagenet-v`, `imagenet-s`, `imagenet` for the ImageNet family).
- `EXPERIMENTS`: names of blocks of `experiments.yaml`. Datasets that are not downloaded are skipped.
- `BACKBONE`: `None` runs every backbone listed in the experiment; `"RN50"` or `"ViT-B/16"` runs only that one.

In [ ]:
# ======================= SETTINGS =======================
REPO_URL = "https://github.com/<your-github-username>/TDA_replication.git"
DATASETS = ["dtd", "caltech101", "eurosat", "oxford_pets"]      # add: fgvc oxford_flowers food101 ucf101 ...
EXPERIMENTS = ["paper_cd"]                                        # e.g. "seeds", "stream_order", "alpha_beta"
BACKBONE = "RN50"                                                 # None = all backbones of the experiment
# ========================================================

import os
from google.colab import drive
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive/TDA_replication"
CODE = "/content/TDA_replication"
os.environ.update({
    "TDA_HOME": CODE,
    "REPO_DIR": f"{CODE}/TDA",                     # official repository
    "DATA_DIR": "/content/data",                   # fast local disk, erased at the end of the session
    "ARCHIVE_DIR": f"{DRIVE_ROOT}/archives",       # downloaded archives, kept on Drive
    "MANUAL_DIR": f"{DRIVE_ROOT}/manual",          # put ImageNet / Stanford Cars files here by hand
    "RESULTS_DIR": f"{DRIVE_ROOT}/results",        # everything to look at
    "LOG_DIR": f"{DRIVE_ROOT}/logs",
    "BACKBONES": BACKBONE or "RN50 ViT-B/16",      # backbones of the equivalence check
    "WANDB_MODE": "disabled",
})
for k in ["ARCHIVE_DIR", "MANUAL_DIR", "RESULTS_DIR", "LOG_DIR", "DATA_DIR"]:
    os.makedirs(os.environ[k], exist_ok=True)

## 3. Code
Clones (or updates) this repository and the official one, and installs the few packages Colab lacks.

In [ ]:
import subprocess
if os.path.isdir(f"{CODE}/.git"):
    !git -C {CODE} pull --ff-only
else:
    !git clone -q {REPO_URL} {CODE}
if not os.path.isdir(f"{CODE}/TDA/.git"):
    !git clone -q https://github.com/kdiAAA/TDA.git {CODE}/TDA
!pip -q install ftfy regex "gdown==4.7.1" wandb
!git -C {CODE} log -1 --format="this repository: %h %s"
!git -C {CODE}/TDA log -1 --format="official code:   %h (%cd)"
%cd {CODE}

## 4. Datasets
Downloads and arranges the datasets in `DATASETS` (minutes for the small ones). Archives under 3 GB are kept on
Drive, so the next session skips the download. ImageNet and Stanford Cars need a manual download: put the files
in `MyDrive/TDA_replication/manual/` (see the README) and add `"imagenet"` / `"stanford_cars"` to `DATASETS`.

In [ ]:
!python prepare_data.py {" ".join(DATASETS)} --keep-archives-under-gb 3

## 5. Check
Counts the test images of every dataset and runs the **equivalence test**: the official code and the
re-implementation on the same 500 images must give identical predictions (`PASS`).

In [ ]:
!python experiments.py check

## 6. Experiments
Each experiment is a block of `experiments.yaml` (open it from the file browser on the left, under
`/content/TDA_replication`). Settings that are already done are skipped, so if the session ends just run
this cell again. To design a new experiment, add a block to `experiments.yaml` in the repository and push it.
`!python experiments.py list` shows what is defined.

In [ ]:
!python experiments.py list
for exp in EXPERIMENTS:
    extra = f"--backbone {BACKBONE}" if BACKBONE else ""
    !python experiments.py run {exp} {extra}

## 7. Report
The same `REPORT.md` as on the cluster: tables vs paper, ablations and every other experiment.
It is saved in `MyDrive/TDA_replication/results/` together with `summary.csv` and the raw results.

In [ ]:
!python experiments.py report --quiet
from IPython.display import Markdown, display
display(Markdown(open(f"{os.environ['RESULTS_DIR']}/REPORT.md", encoding="utf-8").read()))

## Next
Open `notebooks/analysis.ipynb` and set `RESULTS = "/content/drive/MyDrive/TDA_replication/results"` for
plots of the accuracy along the stream, cache dynamics, effect of stream order and seeds, per-class gains.